## Fetching the data

In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("API_KEY_SB")
FRED_LIST= 'fred_daily_series_list.csv'

def get_daily_series(api_key):
    base_url = "https://api.stlouisfed.org/fred/tags/series"
    params = {
        "api_key": api_key,
        "tag_names": "daily",
        "file_type": "json",
        "limit": 1000,
        "offset": 0
    }

    all_series = []

    while True:
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            data = response.json()

            if "seriess" in data:
                series_chunk = data["seriess"]
                all_series.extend(series_chunk)

                if len(series_chunk) < params["limit"]:
                    break
                params["offset"] += params["limit"]
            else:
                break
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
    return all_series



daily_series = get_daily_series(API_KEY)
df = pd.DataFrame(daily_series)
print(f"\nNumber of FRED series with daily data: {len(df)}\n")
# df = df[['id']]
df.to_csv(FRED_LIST, index=False)



Number of FRED series with daily data: 11472



There are different series with daily data, which is significant. We’ll fetch the actual series one by one for years of data. You can adjust the date range according to your preferences by changing START_DATE and END_DATE. Each series will be saved as a CSV file in the folder data.

In [3]:
from io import StringIO

START_DATE = '1970-01-01'
END_DATE   = '2024-12-12'
DATA_FOLDER = 'data_FRED'

In [ ]:
def fetch_data(series_id):
    request = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    request += f"&cosd={START_DATE}"
    request += f"&coed={END_DATE}"
    try:
        response = requests.get(request)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), parse_dates=True)
        # df.rename(
        #     columns={
        #         df.columns[0]: 'ds',
        #         df.columns[1]: 'value',
        #     },
        #     inplace=True,
        # )
        return series_id, df
    except requests.RequestException as e:
        print(f"Error fetching data for {series_id}: {e}")
        return series_id, None

df = pd.read_csv(FRED_LIST)
os.makedirs(DATA_FOLDER, exist_ok=True)
series_ids = df['id'].tolist()

for series_id in series_ids:
    series_id, data = fetch_data(series_id)
    if data is not None:
        filename = os.path.join(DATA_FOLDER, f"{series_id}.csv")
        data.to_csv(filename, index=False)

Error fetching data for IHLIDXAUTPINDUENGI: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error fetching data for NASDAQNQUSB10102015LMT: HTTPSConnectionPool(host='fred.stlouisfed.org', port=443): Max retries exceeded with url: /graph/fredgraph.csv?id=NASDAQNQUSB10102015LMT&cosd=1970-01-01&coed=2024-12-12 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002529087AAE0>, 'Connection to fred.stlouisfed.org timed out. (connect timeout=None)'))


Next, we’ll count the number of CSV files in our data_FRED folder to verify that the API successfully fetched all data series we previously identified.

In [4]:
def count_csv_files(folder_path):
    csv_count = 0
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            csv_count += 1
    return csv_count

num_csv_files = count_csv_files(DATA_FOLDER)
print(f"\nNumber of CSV files in the {DATA_FOLDER} folder: {num_csv_files}\n")


Number of CSV files in the data_FRED folder: 11471



We do a quick exploratory data analysis with one data series.

In [5]:
file_path = os.path.join(DATA_FOLDER, 'AAA10Y.csv')
df = pd.read_csv(file_path)

print("\nLast 10 rows of AAA10Y:")
print("=" * 30)
print(df.tail(10))
print("=" * 30)


Last 10 rows of AAA10Y:
      observation_date  AAA10Y
10934       2024-11-29    0.83
10935       2024-12-02    0.79
10936       2024-12-03    0.79
10937       2024-12-04    0.78
10938       2024-12-05    0.80
10939       2024-12-06    0.81
10940       2024-12-09    0.82
10941       2024-12-10    0.81
10942       2024-12-11    0.84
10943       2024-12-12    0.84


## Preparing the data
1. Determining NYSE trading days
We define a function that returns the dates when the New York Stock Exchange (NYSE) is open.

In [6]:
import pandas_market_calendars as mcal
from typing import Optional

stock_exchange = 'NYSE'
def obtain_market_dates(start_date: str, end_date: str, market : Optional[str] ) -> pd.DataFrame:
    se = mcal.get_calendar(market)
    market_open_dates = se.schedule(
        start_date=start_date,
        end_date=end_date,
    )
    return market_open_dates

market_dates = obtain_market_dates(START_DATE,END_DATE, market=stock_exchange)
print(market_dates.head())
print(market_dates.tail())
print(market_dates.shape)
market_dates.to_csv(f'market_dates_{stock_exchange}.csv')

                         market_open              market_close
1970-01-02 1970-01-02 15:00:00+00:00 1970-01-02 20:00:00+00:00
1970-01-05 1970-01-05 15:00:00+00:00 1970-01-05 20:00:00+00:00
1970-01-06 1970-01-06 15:00:00+00:00 1970-01-06 20:00:00+00:00
1970-01-07 1970-01-07 15:00:00+00:00 1970-01-07 20:00:00+00:00
1970-01-08 1970-01-08 15:00:00+00:00 1970-01-08 20:00:00+00:00
                         market_open              market_close
2024-12-06 2024-12-06 14:30:00+00:00 2024-12-06 21:00:00+00:00
2024-12-09 2024-12-09 14:30:00+00:00 2024-12-09 21:00:00+00:00
2024-12-10 2024-12-10 14:30:00+00:00 2024-12-10 21:00:00+00:00
2024-12-11 2024-12-11 14:30:00+00:00 2024-12-11 21:00:00+00:00
2024-12-12 2024-12-12 14:30:00+00:00 2024-12-12 21:00:00+00:00
(13859, 2)
